# Escadas Rolantes: Manutenção Orientada por Dados
### Dynamic Programming · Challenge Motiva + FIAP 2026/2027

Este notebook cobre as quatro partes pedidas na disciplina:

1. **Degradação acumulada** de cada escada, em três versões (recursiva pura, memoizada e tabulação), com medição de tempo e ranking de criticidade.
2. **Rede de estações como grafo**: lista de adjacência, BFS (menos trechos), Dijkstra (menor tempo) e reconstrução das rotas.
3. **Otimização com A\***: heurística justificada e comparação com o Dijkstra.
4. **Plano de manutenção**: a função `planejar_turno(base, horas)`.

**Arquivos de dados** (pasta `dados/`):
- `ecr_09-26.XLSX`: dataset fornecido pela Motiva, com as notas de manutenção das escadas rolantes da Linha 4-Amarela.
- `estacoes.csv`: vértices do grafo (estações e base de manutenção, com coordenadas aproximadas).
- `trechos.csv`: arestas do grafo (trechos com tempo de deslocamento em minutos).

In [3]:
import time
from collections import deque
import pandas as pd


pd.set_option('display.max_colwidth', None)   # mostra as rotas inteiras nas tabelas

## 1. Preparação do dataset

O dataset da Motiva não traz leituras contínuas de sensores. Ele traz **notas de manutenção** do sistema SAP: cada linha é uma ocorrência registrada em uma escada (alarme, falha, inspeção), com data, sintoma, causa e se a escada ficou indisponível.

Por isso, neste trabalho, **cada nota é tratada como uma "leitura"** do estado da escada. A sequência de notas de uma escada, em ordem de data, é a série sobre a qual a degradação é acumulada.

Passos de preparação:
1. Remover notas **canceladas** (status com `CANC`), que não representam uma ocorrência real.
2. Extrair a **estação** e a **escada** do código do local de instalação. Exemplo: `SP04-6-ECR-PIN-ER01` → estação `PIN`, escada `PIN-ER01`.
3. Manter apenas notas a partir de **2024**, período em que o registro é mais completo e que contém as **197 escadas** citadas pela Motiva.

In [4]:
notas = pd.read_excel('dados/ecr_09-26.XLSX', sheet_name='ZMM_ANALISE_NOTAS')
print('Notas no arquivo original:', len(notas))

# 1) remove notas canceladas
canceladas = notas['Status do Usuário da Ordem'].fillna('').str.contains('CANC')
notas = notas[~canceladas]

# 2) separa o código do local: SP04-6-ECR-PIN-ER01
partes = notas['Local de Instalação'].str.split('-')
notas = notas[partes.str.len() == 5]
partes = notas['Local de Instalação'].str.split('-')
notas['estacao'] = partes.str[3]
notas['escada'] = partes.str[3] + '-' + partes.str[4]

# 3) converte a data e mantém o período a partir de 2024
notas['data'] = pd.to_datetime(notas['Início avaria'], format='%d/%m/%Y', errors='coerce')
notas = notas[notas['data'] >= '2024-01-01']
notas = notas.sort_values(['escada', 'data', 'Nota'])

print('Notas após a preparação:', len(notas))
print('Escadas:', notas['escada'].nunique())
print('Estações:', notas['estacao'].nunique())

Notas no arquivo original: 104119
Notas após a preparação: 27389
Escadas: 197
Estações: 12


### Peso de cada leitura

Cada nota recebe um **peso de desgaste**. Quanto mais grave a ocorrência, maior o peso:

| Situação | Peso |
|---|---|
| Toda ocorrência registrada | +1 |
| Nota marcada como **Falha** | +2 |
| Escada ficou **Indisponível** | +3 |
| Sintoma ligado a um dos 5 sinais da Motiva | +2 ou +3 |

Os sintomas foram associados aos sinais que a Motiva apresentou no Challenge (vibração, temperatura, corrente/tensão, nível de óleo e velocidade), porque são os comportamentos que esses sensores detectariam.

In [5]:
# sintoma do dataset -> peso extra (sinal monitorado entre parênteses)
PESO_SINTOMA = {
    'RUIDO ANORMAL': 2,                  # vibração
    'FOLGA ANORMAL': 2,                  # vibração
    'QUEIMADO(A)': 3,                    # temperatura
    'ALARME-CORRENTE PRINCIPAL': 3,      # corrente/tensão
    'NAO LIGA': 2,                       # corrente/tensão
    'ALARME-BOMBA DE LUBRIFICACAO': 2,   # nível de óleo
    'VAZAMENTO': 2,                      # nível de óleo
    'ALARME-SUB/SOBREVELOCIDADE': 3,     # velocidade
    'TRAVADO(A)': 3,                     # velocidade
}


def calcular_peso(falha, indisponivel, sintoma):
    """Calcula o peso de desgaste de uma nota de manutenção.

    Parâmetros:
        falha: 'X' se a nota foi classificada como falha.
        indisponivel: 'X' se a escada ficou fora de operação.
        sintoma: texto do sintoma registrado.

    Retorna:
        Número inteiro com o peso da ocorrência.
    """
    peso = 1
    if falha == 'X':
        peso = peso + 2
    if indisponivel == 'X':
        peso = peso + 3
    if sintoma in PESO_SINTOMA:
        peso = peso + PESO_SINTOMA[sintoma]
    return peso


pesos = []
for indice, linha in notas.iterrows():
    pesos.append(calcular_peso(linha['Falha'], linha['Indisponível'], linha['Sintoma']))
notas['peso'] = pesos

# leituras[escada] = lista de pesos em ordem de data
leituras = {}
estacao_da_escada = {}
for escada, grupo in notas.groupby('escada'):
    leituras[escada] = list(grupo['peso'])
    estacao_da_escada[escada] = list(grupo['estacao'])[0]

print('Exemplo - PIN-ER01, primeiras 15 leituras:', leituras['PIN-ER01'][:15])

Exemplo - PIN-ER01, primeiras 15 leituras: [6, 1, 4, 4, 3, 3, 6, 1, 1, 4, 6, 4, 1, 4, 3]


## 2. Degradação acumulada: três versões

A degradação depois da leitura `i` depende da degradação depois da leitura anterior:

```
D(i) = D(i - 1) × FATOR + peso(i)        com D(-1) = 0
```

O **fator de esquecimento** (0,99) faz as ocorrências antigas pesarem um pouco menos que as recentes. Uma escada que falhou muito há dois anos e foi reparada não deve ser tão crítica quanto uma que está falhando agora.

O objetivo é obter a **curva de degradação** de cada escada, ou seja, o valor de `D(i)` para **todas** as leituras. É aqui que as três versões se diferenciam:

- **Recursiva pura:** para calcular cada ponto da curva, refaz toda a recursão desde a primeira leitura. Para `n` leituras, faz cerca de `n²/2` chamadas.
- **Memoizada:** a mesma recursão, mas guarda cada `D(i)` em um dicionário. Cada valor é calculado uma única vez: `n` chamadas.
- **Tabulação:** sem recursão. Preenche uma lista do início ao fim, usando o valor anterior que já está na tabela: `n` passos.

In [6]:
FATOR = 0.99


def degradacao_recursiva(pesos, i):
    """Degradação acumulada até a leitura i, por recursão pura."""
    if i < 0:
        return 0
    return degradacao_recursiva(pesos, i - 1) * FATOR + pesos[i]


def curva_recursiva(pesos):
    """Curva completa usando a recursão pura (recalcula tudo a cada ponto)."""
    curva = []
    for i in range(len(pesos)):
        curva.append(degradacao_recursiva(pesos, i))
    return curva


def degradacao_memoizada(pesos, i, memo):
    """Degradação até a leitura i, guardando os resultados no dicionário memo."""
    if i < 0:
        return 0
    if i in memo:
        return memo[i]
    memo[i] = degradacao_memoizada(pesos, i - 1, memo) * FATOR + pesos[i]
    return memo[i]


def curva_memoizada(pesos):
    """Curva completa usando recursão com memoização."""
    memo = {}
    curva = []
    for i in range(len(pesos)):
        curva.append(degradacao_memoizada(pesos, i, memo))
    return curva


def curva_tabulada(pesos):
    """Curva completa usando tabulação (iterativa, de baixo para cima)."""
    tabela = [0] * len(pesos)
    tabela[0] = pesos[0]
    for i in range(1, len(pesos)):
        tabela[i] = tabela[i - 1] * FATOR + pesos[i]
    return tabela

### Medição das três versões

Cada versão calcula a curva de **todas as 197 escadas**. O tempo é medido com `time.time()`.

In [7]:
def medir(funcao_curva):
    """Executa uma versão para todas as escadas e mede o tempo total.

    Retorna:
        (dicionário escada -> curva, tempo em segundos)
    """
    inicio = time.time()
    curvas = {}
    for escada in leituras:
        curvas[escada] = funcao_curva(leituras[escada])
    fim = time.time()
    return curvas, fim - inicio


curvas_rec, tempo_rec = medir(curva_recursiva)
curvas_memo, tempo_memo = medir(curva_memoizada)
curvas_tab, tempo_tab = medir(curva_tabulada)

print(f'Recursiva pura : {tempo_rec:.4f} s')
print(f'Memoizada      : {tempo_memo:.4f} s')
print(f'Tabulação      : {tempo_tab:.4f} s')
print(f'A recursiva pura foi {tempo_rec / tempo_tab:.0f} vezes mais lenta que a tabulação.')

Recursiva pura : 0.2262 s
Memoizada      : 0.0067 s
Tabulação      : 0.0025 s
A recursiva pura foi 91 vezes mais lenta que a tabulação.


In [8]:
# as três versões precisam chegar exatamente nos mesmos valores
diferentes = 0
for escada in leituras:
    for i in range(len(leituras[escada])):
        a = round(curvas_rec[escada][i], 6)
        b = round(curvas_memo[escada][i], 6)
        c = round(curvas_tab[escada][i], 6)
        if a != b or b != c:
            diferentes = diferentes + 1

print('Pontos de curva comparados:', len(notas))
print('Pontos com resultados diferentes:', diferentes)

Pontos de curva comparados: 27389
Pontos com resultados diferentes: 0


### Crescimento do tempo conforme o número de leituras

Para ver o comportamento de cada versão, a escada com mais leituras é processada com 50, 100, 200 e 400 leituras.

In [9]:
escada_maior = notas['escada'].value_counts().index[0]
pesos_maior = leituras[escada_maior]
print('Escada com mais leituras:', escada_maior, '-', len(pesos_maior), 'leituras')

tamanhos = [50, 100, 200, 400]
linhas = []
for n in tamanhos:
    trecho = pesos_maior[:n]
    resultado = [n]
    for funcao in [curva_recursiva, curva_memoizada, curva_tabulada]:
        inicio = time.time()
        for repeticao in range(20):
            funcao(trecho)
        resultado.append((time.time() - inicio) / 20 * 1000)
    linhas.append(resultado)

crescimento = pd.DataFrame(linhas, columns=['leituras', 'recursiva (ms)', 'memoizada (ms)', 'tabulação (ms)'])
crescimento.round(3)

Escada com mais leituras: PIN-ER24 - 447 leituras


,leituras,recursiva (ms),memoizada (ms),tabulação (ms)
0,50,0.085,0.011,0.003
1,100,0.322,0.022,0.006
2,200,1.777,0.045,0.015
3,400,8.560,0.090,0.026


### Interpretação

- **As três versões chegam exatamente aos mesmos valores** em todos os pontos de curva. Memoização e tabulação não mudam o resultado, só o custo de calcular.
- **A recursiva pura é dezenas de vezes mais lenta.** Para montar a curva, ela refaz `D(0), D(1), ..., D(i-1)` a cada novo ponto. Com 447 leituras, isso dá cerca de 100 mil chamadas, contra 447 da versão memoizada.
- **O crescimento confirma a teoria.** Quando o número de leituras é multiplicado por 8 (de 50 para 400), o tempo da recursiva pura cresce muito mais que 8 vezes, perto de 8² = 64 (crescimento **quadrático**). Já memoizada e tabulação crescem perto de 8 vezes (crescimento **linear**).
- **A tabulação é a mais rápida e a mais segura.** Ela não paga o custo de chamar funções e não depende da pilha de recursão. O Python interrompe recursões com mais de cerca de 1000 chamadas aninhadas. Hoje a maior escada tem 447 leituras, mas com alguns anos a mais de histórico as duas versões recursivas parariam de funcionar. Por isso, a tabulação é a versão usada no ranking.

### Ranking de criticidade

A criticidade de cada escada é a **degradação acumulada no fim da série**, ou seja, o último ponto da curva.

In [10]:
lista_escadas = []
lista_estacoes = []
lista_leituras = []
lista_degradacao = []
for escada in curvas_tab:
    lista_escadas.append(escada)
    lista_estacoes.append(estacao_da_escada[escada])
    lista_leituras.append(len(leituras[escada]))
    lista_degradacao.append(round(curvas_tab[escada][-1], 1))

ranking = pd.DataFrame({
    'escada': lista_escadas,
    'estacao': lista_estacoes,
    'leituras': lista_leituras,
    'degradacao': lista_degradacao,
})
ranking = ranking.sort_values('degradacao', ascending=False).reset_index(drop=True)
ranking.index = ranking.index + 1
ranking.head(15)

,escada,estacao,leituras,degradacao
1,PTA-ER04,PTA,230,411.0
2,PIN-ER05,PIN,359,385.8
3,LUZ-ER04,LUZ,341,381.9
4,PIN-ER32,PIN,247,379.1
5,PIN-ER02,PIN,222,376.1
6,PIN-ER24,PIN,447,373.5
7,PIN-ER28,PIN,251,368.4
8,PIN-ER06,PIN,204,366.2
9,LUZ-ER02,LUZ,363,365.5
10,PIN-ER14,PIN,238,362.8


In [11]:
# degradação média das escadas por estação
ranking.groupby('estacao')['degradacao'].mean().round(1).sort_values(ascending=False)

estacao
PIN    304.1
REP    301.4
LUZ    294.0
PTA    289.4
BUT    283.5
FAL    281.6
MBI    244.4
VSO    223.2
FRA    218.6
FRE    202.7
HIG    194.9
TVS    160.9
Name: degradacao, dtype: float64

### Interpretação

**Observações dos dados:**
- **PTA-ER04 (Paulista) é a escada mais crítica**, mesmo tendo menos leituras (230) que várias outras. A **PIN-ER24**, com o maior número de notas (447), aparece só em 6º lugar. Isso mostra que a criticidade não é só "quantidade de notas": a gravidade (falha, indisponibilidade, sintoma) e o quanto as ocorrências são recentes pesam no resultado.
- **Pinheiros concentra 7 das 15 escadas mais críticas** e tem a maior degradação média por estação. Em seguida vêm República, Luz e Paulista.
- **O Terminal Vila Sônia tem a menor degradação média.**

**Hipótese técnica (não comprovada pelo dataset):** Pinheiros, República, Luz e Paulista são estações de **integração** com outras linhas. O maior fluxo de passageiros pode explicar o desgaste maior das escadas. O dataset não tem dados de fluxo para confirmar isso.

**Limitação:** o ranking depende dos pesos escolhidos pelo grupo (tabela da seção 1). Eles são uma premissa e deveriam ser validados com a equipe de manutenção da Motiva.

## 3. Rede de estações como grafo

**Vértices:** as 12 estações da Linha 4-Amarela (operada pela ViaQuatro, do grupo Motiva) que aparecem no dataset, mais a **base de manutenção** no Pátio Vila Sônia.

**Arestas:** trechos com o tempo de deslocamento da equipe em minutos. Há três modos:
- **metrô:** entre estações vizinhas da linha (rápido, mas um trecho por estação);
- **a pé:** ligações curtas da base e do terminal de ônibus;
- **van:** atalhos pelas ruas (Base → Butantã, Base → Pinheiros, Pinheiros → Paulista, Paulista → Luz). Pulam várias estações, mas pegam trânsito.

Os tempos são estimativas da equipe para o trabalho. O grafo é **não direcionado**: o tempo de ida e de volta em um trecho é o mesmo.

Cada escada do dataset está ligada a uma estação pelo código do local (`PIN-ER01` → `PIN`).

In [12]:
estacoes = pd.read_csv('dados/estacoes.csv')
trechos = pd.read_csv('dados/trechos.csv')

# lista de adjacência: grafo[estacao] = [(vizinha, minutos), ...]
grafo = {}
for sigla in estacoes['sigla']:
    grafo[sigla] = []
for indice, trecho in trechos.iterrows():
    grafo[trecho['origem']].append((trecho['destino'], trecho['minutos']))
    grafo[trecho['destino']].append((trecho['origem'], trecho['minutos']))

nome_da_estacao = {}
for indice, estacao in estacoes.iterrows():
    nome_da_estacao[estacao['sigla']] = estacao['nome']

for sigla in grafo:
    print(sigla, '->', grafo[sigla])

# confere se toda escada está ligada a uma estação do grafo
sem_estacao = 0
for escada in estacao_da_escada:
    if estacao_da_escada[escada] not in grafo:
        sem_estacao = sem_estacao + 1
print()
print('Escadas sem estação no grafo:', sem_estacao)

BASE -> [('VSO', 6), ('TVS', 5), ('BUT', 15), ('PIN', 22)]
LUZ -> [('REP', 2), ('PTA', 20)]
REP -> [('LUZ', 2), ('HIG', 2)]
HIG -> [('REP', 2), ('PTA', 2)]
PTA -> [('HIG', 2), ('FRE', 2), ('PIN', 18), ('LUZ', 20)]
FRE -> [('PTA', 2), ('FRA', 2)]
FRA -> [('FRE', 2), ('FAL', 2)]
FAL -> [('FRA', 2), ('PIN', 2)]
PIN -> [('FAL', 2), ('BUT', 3), ('BASE', 22), ('PTA', 18)]
BUT -> [('PIN', 3), ('MBI', 3), ('BASE', 15)]
MBI -> [('BUT', 3), ('VSO', 3)]
VSO -> [('MBI', 3), ('TVS', 4), ('BASE', 6)]
TVS -> [('VSO', 4), ('BASE', 5)]

Escadas sem estação no grafo: 0


### BFS: caminho com menos trechos

A BFS visita as estações em "camadas": primeiro as vizinhas da base, depois as vizinhas das vizinhas, e assim por diante, usando uma **fila** (`deque`). A primeira vez que uma estação é alcançada é pelo caminho com menos trechos. A BFS **ignora o tempo** de cada trecho.

O dicionário `anterior` guarda de onde cada estação foi alcançada. A rota é reconstruída voltando do destino até a origem.

In [13]:
def bfs(grafo, origem):
    """Busca em largura a partir da origem.

    Retorna:
        Dicionário anterior[estacao] = estação de onde ela foi alcançada.
    """
    anterior = {origem: None}
    fila = deque([origem])
    while len(fila) > 0:
        atual = fila.popleft()
        for vizinha, minutos in grafo[atual]:
            if vizinha not in anterior:
                anterior[vizinha] = atual
                fila.append(vizinha)
    return anterior


def reconstruir_rota(anterior, destino):
    """Monta a rota da origem até o destino seguindo o dicionário anterior."""
    rota = []
    atual = destino
    while atual is not None:
        rota.append(atual)
        atual = anterior[atual]
    rota.reverse()
    return rota


def tempo_da_rota(rota):
    """Soma os minutos dos trechos de uma rota."""
    total = 0
    for i in range(len(rota) - 1):
        for vizinha, minutos in grafo[rota[i]]:
            if vizinha == rota[i + 1]:
                total = total + minutos
    return total

### Dijkstra: caminho de menor tempo

O Dijkstra mantém a menor distância conhecida até cada estação. A cada passo, escolhe a estação **ainda não visitada com a menor distância** e tenta melhorar o caminho até as vizinhas dela.

Como a rede é pequena (13 vértices), a escolha da menor distância é feita percorrendo a lista de estações, sem estruturas extras. Se for passado um `destino`, a busca para assim que ele é visitado. Isso permite comparar com o A\* mais à frente. A função também conta quantas estações foram **expandidas**.

In [14]:
INFINITO = float('inf')


def dijkstra(grafo, origem, destino=None):
    """Caminho de menor tempo a partir da origem.

    Parâmetros:
        grafo: lista de adjacência.
        origem: estação de partida.
        destino: se informado, para a busca ao chegar nele.

    Retorna:
        (distancia, anterior, expandidas)
    """
    distancia = {}
    for estacao in grafo:
        distancia[estacao] = INFINITO
    distancia[origem] = 0
    anterior = {origem: None}
    visitadas = []
    expandidas = 0

    while len(visitadas) < len(grafo):
        # escolhe a estação não visitada com menor distância
        atual = None
        for estacao in grafo:
            if estacao not in visitadas:
                if atual is None or distancia[estacao] < distancia[atual]:
                    atual = estacao
        visitadas.append(atual)
        if atual == destino:
            break
        expandidas = expandidas + 1

        for vizinha, minutos in grafo[atual]:
            nova_distancia = distancia[atual] + minutos
            if nova_distancia < distancia[vizinha]:
                distancia[vizinha] = nova_distancia
                anterior[vizinha] = atual

    return distancia, anterior, expandidas

### Rotas da base até cada estação

In [15]:
BASE = 'BASE'

anterior_bfs = bfs(grafo, BASE)
distancia_dij, anterior_dij, expandidas = dijkstra(grafo, BASE)

linhas = []
for estacao in grafo:
    if estacao != BASE:
        rota_bfs = reconstruir_rota(anterior_bfs, estacao)
        rota_dij = reconstruir_rota(anterior_dij, estacao)
        linhas.append([
            estacao,
            ' > '.join(rota_bfs), len(rota_bfs) - 1, tempo_da_rota(rota_bfs),
            ' > '.join(rota_dij), len(rota_dij) - 1, distancia_dij[estacao],
        ])

rotas = pd.DataFrame(linhas, columns=[
    'destino',
    'rota BFS', 'trechos BFS', 'min BFS',
    'rota Dijkstra', 'trechos Dijkstra', 'min Dijkstra',
])
rotas

,destino,rota BFS,trechos BFS,min BFS,rota Dijkstra,trechos Dijkstra,min Dijkstra
0,LUZ,BASE > PIN > PTA > LUZ,3,60,BASE > VSO > MBI > BUT > PIN > FAL > FRA > FRE > PTA > HIG > REP > LUZ,11,29
1,REP,BASE > PIN > PTA > HIG > REP,4,44,BASE > VSO > MBI > BUT > PIN > FAL > FRA > FRE > PTA > HIG > REP,10,27
2,HIG,BASE > PIN > PTA > HIG,3,42,BASE > VSO > MBI > BUT > PIN > FAL > FRA > FRE > PTA > HIG,9,25
3,PTA,BASE > PIN > PTA,2,40,BASE > VSO > MBI > BUT > PIN > FAL > FRA > FRE > PTA,8,23
4,FRE,BASE > PIN > PTA > FRE,3,42,BASE > VSO > MBI > BUT > PIN > FAL > FRA > FRE,7,21
5,FRA,BASE > PIN > FAL > FRA,3,26,BASE > VSO > MBI > BUT > PIN > FAL > FRA,6,19
6,FAL,BASE > PIN > FAL,2,24,BASE > VSO > MBI > BUT > PIN > FAL,5,17
7,PIN,BASE > PIN,1,22,BASE > VSO > MBI > BUT > PIN,4,15
8,BUT,BASE > BUT,1,15,BASE > VSO > MBI > BUT,3,12
9,MBI,BASE > VSO > MBI,2,9,BASE > VSO > MBI,2,9


### Interpretação

- **BFS e Dijkstra escolhem caminhos bem diferentes.** A BFS sempre prefere os atalhos de **van**, porque eles pulam várias estações em um só trecho. O Dijkstra sempre prefere o **metrô**, porque os trechos são curtos em tempo.
- **Exemplo da Luz:** a BFS chega em **3 trechos, mas leva 60 min** (Base → Pinheiros → Paulista → Luz, tudo de van). O Dijkstra usa **11 trechos e leva só 29 min**, pelo metrô.
- **Conclusão:** "menos trechos" não significa "menos tempo" quando os trechos têm pesos diferentes. Para planejar o deslocamento da equipe, o Dijkstra é o algoritmo certo. A BFS só seria útil se o objetivo fosse reduzir trocas de meio de transporte.
- Para Vila Sônia, Terminal Vila Sônia e Morumbi, os dois algoritmos coincidem, porque essas estações estão perto da base.
- Com os tempos atuais, **as vans nunca entram no caminho de menor tempo**. Elas passariam a ser a melhor opção se o metrô estivesse parado (por exemplo, fora do horário de operação), bastando aumentar o peso dos trechos de metrô no `trechos.csv`.

## 4. Otimização de rotas com A\*

O A\* funciona como o Dijkstra, mas escolhe a próxima estação pelo valor

```
f(n) = g(n) + h(n)
```

- `g(n)`: tempo real já gasto da origem até `n`;
- `h(n)`: **estimativa** do tempo que falta de `n` até o destino (a heurística).

Assim, ele dá preferência a estações que estão "na direção" do destino e evita explorar o lado oposto da rede.

### Heurística escolhida e justificativa

```
h(n) = distância em linha reta(n, destino) / VELOCIDADE_MÁXIMA
```

- A **distância em linha reta** é calculada pelas coordenadas das estações (convertendo graus de latitude/longitude para km).
- A **VELOCIDADE_MÁXIMA** é a maior velocidade encontrada entre todos os trechos da rede (distância em linha reta do trecho ÷ minutos do trecho).

**Por que essa heurística é válida (admissível):** nenhum caminho real é mais curto que a linha reta, e nenhum trecho é percorrido mais rápido que a velocidade máxima da rede. Portanto `h(n)` **nunca superestima** o tempo real que falta. Com uma heurística admissível, o A\* garante encontrar o mesmo caminho ótimo que o Dijkstra.

In [16]:
latitude = {}
longitude = {}
for indice, estacao in estacoes.iterrows():
    latitude[estacao['sigla']] = estacao['latitude']
    longitude[estacao['sigla']] = estacao['longitude']


def distancia_km(a, b):
    """Distância aproximada em linha reta entre duas estações, em km.

    1 grau de latitude ≈ 111 km; em São Paulo (latitude -23,5°),
    1 grau de longitude ≈ 101,8 km.
    """
    dy = (latitude[a] - latitude[b]) * 111.0
    dx = (longitude[a] - longitude[b]) * 101.8
    return (dx ** 2 + dy ** 2) ** 0.5


# maior velocidade (km/min) entre todos os trechos da rede
VELOCIDADE_MAXIMA = 0
for indice, trecho in trechos.iterrows():
    velocidade = distancia_km(trecho['origem'], trecho['destino']) / trecho['minutos']
    if velocidade > VELOCIDADE_MAXIMA:
        VELOCIDADE_MAXIMA = velocidade

print(f'Velocidade máxima da rede: {VELOCIDADE_MAXIMA:.2f} km/min ({VELOCIDADE_MAXIMA * 60:.0f} km/h)')


def heuristica(estacao, destino):
    """Estimativa otimista (em minutos) do tempo até o destino."""
    return distancia_km(estacao, destino) / VELOCIDADE_MAXIMA


def a_estrela(grafo, origem, destino):
    """Caminho de menor tempo da origem ao destino usando A*.

    Retorna:
        (tempo, anterior, expandidas)
    """
    custo = {origem: 0}
    anterior = {origem: None}
    abertas = [origem]
    expandidas = 0

    while len(abertas) > 0:
        # escolhe a estação aberta com menor custo + heurística
        atual = abertas[0]
        for estacao in abertas:
            if custo[estacao] + heuristica(estacao, destino) < custo[atual] + heuristica(atual, destino):
                atual = estacao
        if atual == destino:
            break
        abertas.remove(atual)
        expandidas = expandidas + 1

        for vizinha, minutos in grafo[atual]:
            novo_custo = custo[atual] + minutos
            if vizinha not in custo or novo_custo < custo[vizinha]:
                custo[vizinha] = novo_custo
                anterior[vizinha] = atual
                if vizinha not in abertas:
                    abertas.append(vizinha)

    return custo[destino], anterior, expandidas

Velocidade máxima da rede: 0.74 km/min (45 km/h)


### Comparação A\* × Dijkstra

Para cada destino, partindo da base: tempo encontrado, rota e número de estações **expandidas** (exploradas) até chegar no destino.

In [17]:
linhas = []
for estacao in grafo:
    if estacao != BASE:
        dist_d, ant_d, exp_d = dijkstra(grafo, BASE, estacao)
        tempo_a, ant_a, exp_a = a_estrela(grafo, BASE, estacao)
        linhas.append([
            estacao,
            dist_d[estacao], exp_d,
            tempo_a, exp_a,
            reconstruir_rota(ant_d, estacao) == reconstruir_rota(ant_a, estacao),
        ])

comparacao = pd.DataFrame(linhas, columns=[
    'destino', 'min Dijkstra', 'expandidas Dijkstra', 'min A*', 'expandidas A*', 'mesma rota',
])
comparacao

,destino,min Dijkstra,expandidas Dijkstra,min A*,expandidas A*,mesma rota
0,LUZ,29,12,29,12,True
1,REP,27,11,27,11,True
2,HIG,25,10,25,10,True
3,PTA,23,9,23,9,True
4,FRE,21,8,21,8,True
5,FRA,19,7,19,7,True
6,FAL,17,6,17,6,True
7,PIN,15,5,15,5,True
8,BUT,12,4,12,4,True
9,MBI,9,3,9,3,True


In [18]:
# comparação entre todos os pares origem-destino da rede
total_dij = 0
total_a = 0
iguais = 0
pares = 0
for origem in grafo:
    for destino in grafo:
        if origem != destino:
            dist_d, ant_d, exp_d = dijkstra(grafo, origem, destino)
            tempo_a, ant_a, exp_a = a_estrela(grafo, origem, destino)
            total_dij = total_dij + exp_d
            total_a = total_a + exp_a
            pares = pares + 1
            if dist_d[destino] == tempo_a:
                iguais = iguais + 1

print('Pares origem-destino testados:', pares)
print('Pares com o mesmo tempo ótimo:', iguais)
print('Estações expandidas - Dijkstra:', total_dij)
print('Estações expandidas - A*      :', total_a)
print(f'Redução de exploração com A*: {(1 - total_a / total_dij) * 100:.1f}%')

# tempo de execução (repetido para ser mensurável)
REPETICOES = 200
inicio = time.time()
for repeticao in range(REPETICOES):
    for origem in grafo:
        for destino in grafo:
            if origem != destino:
                dijkstra(grafo, origem, destino)
tempo_dij = time.time() - inicio

inicio = time.time()
for repeticao in range(REPETICOES):
    for origem in grafo:
        for destino in grafo:
            if origem != destino:
                a_estrela(grafo, origem, destino)
tempo_a = time.time() - inicio

print()
print(f'Tempo Dijkstra ({REPETICOES}x todos os pares): {tempo_dij:.3f} s')
print(f'Tempo A*       ({REPETICOES}x todos os pares): {tempo_a:.3f} s')

Pares origem-destino testados: 156
Pares com o mesmo tempo ótimo: 156
Estações expandidas - Dijkstra: 1014
Estações expandidas - A*      : 790
Redução de exploração com A*: 22.1%

Tempo Dijkstra (200x todos os pares): 0.362 s
Tempo A*       (200x todos os pares): 0.475 s


In [19]:
# exemplo com a origem no meio da linha: Faria Lima -> Luz
dist_d, ant_d, exp_d = dijkstra(grafo, 'FAL', 'LUZ')
tempo_a, ant_a, exp_a = a_estrela(grafo, 'FAL', 'LUZ')
print('Rota:', ' > '.join(reconstruir_rota(ant_a, 'LUZ')), '-', tempo_a, 'min')
print('Dijkstra expandiu', exp_d, 'estações')
print('A* expandiu      ', exp_a, 'estações')

Rota: FAL > FRA > FRE > PTA > HIG > REP > LUZ - 12 min
Dijkstra expandiu 10 estações
A* expandiu       6 estações


### Interpretação

- **O A\* encontrou o mesmo tempo ótimo que o Dijkstra nos 156 pares** de origem e destino. Isso confirma na prática que a heurística é admissível.
- **Partindo da base, o A\* não economiza nada.** A base fica na ponta da Linha 4, então todas as estações mais próximas que o destino estão justamente no caminho até ele. Não há "lado errado" para evitar.
- **Com a origem no meio da linha, o A\* explora menos.** De Faria Lima até Luz, o Dijkstra também explora estações na direção de Butantã e Vila Sônia, porque estão perto em tempo. O A\* percebe que elas se afastam da Luz e as deixa de lado. Considerando todos os pares, o A\* expandiu cerca de **22% menos estações**.
- **Mesmo assim, o tempo de execução é parecido.** O A\* gasta tempo calculando a heurística a cada comparação, e em uma rede de 13 vértices isso empata com a economia de exploração. O ganho real do A\* aparece em redes grandes, por exemplo se todas as linhas operadas pela Motiva fossem incluídas no grafo.

## 5. Plano de manutenção: `planejar_turno(base, horas)`

A função junta o **ranking de criticidade** com as **rotas de menor tempo**:

1. A equipe sai da base com o tempo do turno disponível.
2. Percorre o ranking da escada mais crítica para a menos crítica.
3. Para cada escada, calcula com Dijkstra o tempo da posição atual da equipe até a estação da escada **e** o tempo para voltar dessa estação até a base.
4. A escada só entra no plano se `ida + atendimento + volta à base` ainda couber no tempo restante. Isso garante que a equipe sempre consegue terminar o turno na base.
5. Se a escada não cabe, ela é pulada e a próxima do ranking é testada.

O tempo de atendimento é fixo em **60 minutos** por escada (premissa ajustável).

In [20]:
TEMPO_ATENDIMENTO = 60


def planejar_turno(base, horas):
    """Monta o plano de atendimento de um turno.

    Parâmetros:
        base: estação de onde a equipe sai e para onde volta.
        horas: duração do turno em horas.

    Retorna:
        DataFrame com a ordem de atendimento, a escada, a rota e os tempos.
    """
    tempo_turno = horas * 60
    tempo_usado = 0
    posicao = base
    plano = []

    for i in ranking.index:
        escada = ranking['escada'][i]
        estacao = estacao_da_escada[escada]

        distancia_ida, anterior_ida, exp = dijkstra(grafo, posicao)
        ida = distancia_ida[estacao]
        distancia_volta, anterior_volta, exp = dijkstra(grafo, estacao)
        volta = distancia_volta[base]

        if tempo_usado + ida + TEMPO_ATENDIMENTO + volta <= tempo_turno:
            rota = reconstruir_rota(anterior_ida, estacao)
            tempo_usado = tempo_usado + ida + TEMPO_ATENDIMENTO
            plano.append([
                len(plano) + 1, escada, i, ranking['degradacao'][i],
                ' > '.join(rota), ida, tempo_usado,
            ])
            posicao = estacao

    # volta para a base no fim do turno
    distancia_volta, anterior_volta, exp = dijkstra(grafo, posicao)
    rota_volta = reconstruir_rota(anterior_volta, base)
    tempo_usado = tempo_usado + distancia_volta[base]
    plano.append([
        '-', 'retorno à base', '-', '-',
        ' > '.join(rota_volta), distancia_volta[base], tempo_usado,
    ])

    return pd.DataFrame(plano, columns=[
        'ordem', 'escada', 'posição no ranking', 'degradação',
        'rota', 'deslocamento (min)', 'tempo acumulado (min)',
    ])

In [21]:
turno_8h = planejar_turno(BASE, 8)
turno_8h

,ordem,escada,posição no ranking,degradação,rota,deslocamento (min),tempo acumulado (min)
0,1,PTA-ER04,1,411.0,BASE > VSO > MBI > BUT > PIN > FAL > FRA > FRE > PTA,23,83
1,2,PIN-ER05,2,385.8,PTA > FRE > FRA > FAL > PIN,8,151
2,3,LUZ-ER04,3,381.9,PIN > FAL > FRA > FRE > PTA > HIG > REP > LUZ,14,225
3,4,PIN-ER32,4,379.1,LUZ > REP > HIG > PTA > FRE > FRA > FAL > PIN,14,299
4,5,PIN-ER02,5,376.1,PIN,0,359
5,6,PIN-ER24,6,373.5,PIN,0,419
6,-,retorno à base,-,-,PIN > BUT > MBI > VSO > BASE,15,434


In [22]:
turno_6h = planejar_turno(BASE, 6)
turno_6h

,ordem,escada,posição no ranking,degradação,rota,deslocamento (min),tempo acumulado (min)
0,1,PTA-ER04,1,411.0,BASE > VSO > MBI > BUT > PIN > FAL > FRA > FRE > PTA,23,83
1,2,PIN-ER05,2,385.8,PTA > FRE > FRA > FAL > PIN,8,151
2,3,LUZ-ER04,3,381.9,PIN > FAL > FRA > FRE > PTA > HIG > REP > LUZ,14,225
3,4,PIN-ER32,4,379.1,LUZ > REP > HIG > PTA > FRE > FRA > FAL > PIN,14,299
4,-,retorno à base,-,-,PIN > BUT > MBI > VSO > BASE,15,314


### Interpretação

- **Turno de 8 h:** a equipe atende as **6 escadas mais críticas** do ranking, na ordem de prioridade, e volta para a base com **434 dos 480 minutos** usados. A 7ª escada (PIN-ER28) já estaria em Pinheiros, mas não cabe: 419 + 60 de atendimento + 15 de volta = 494 min, mais que o turno.
- **Turno de 6 h:** cabem as **4 primeiras** escadas do ranking.
- **O plano é coerente com os dados:** respeita a ordem de criticidade, usa sempre a rota de menor tempo (Dijkstra) e nunca deixa a equipe sem tempo para voltar à base.
- **Limitação percebida:** seguir a prioridade de forma estrita gera um vai e vem (Pinheiros → Luz → Pinheiros, 28 min só de deslocamento). Uma melhoria seria agrupar escadas da mesma estação quando a diferença de criticidade entre elas for pequena. Isso troca um pouco de prioridade por mais escadas atendidas no turno.

## 6. Conclusões

- O dataset da Motiva traz **notas de manutenção** e não leituras contínuas de sensores. Cada nota foi tratada como uma leitura, e os sintomas foram ligados aos cinco sinais monitorados (vibração, temperatura, corrente/tensão, óleo e velocidade).
- A **programação dinâmica** transforma o cálculo da curva de degradação de quadrático (recursão pura) para linear (memoização e tabulação), com resultados idênticos. A **tabulação** é a melhor opção: mais rápida e sem limite de recursão.
- **Pinheiros** é a estação com mais escadas críticas, e **PTA-ER04** é a escada prioritária.
- Na rede, o **Dijkstra** é o algoritmo adequado para o tempo de deslocamento. A **BFS** minimiza trechos, mas leva a rotas até 2 vezes mais lentas.
- O **A\*** dá o mesmo resultado ótimo que o Dijkstra explorando cerca de 22% menos estações. A vantagem cresce em redes maiores.
- O `planejar_turno` junta tudo em uma decisão prática: **quais escadas atender, em que ordem e por qual caminho**, sempre dentro do turno.

**Próximos passos:** validar os pesos com a Motiva, incluir dados de fluxo de passageiros e testar o agrupamento por estação no plano de turno.